# 09 -- LightGBM (Champion Candidate B)
**Purpose:** Train LightGBM as the second champion candidate. Compare against XGBoost to select the winner.

**Input:** X_train, X_val (data/processed/)
**Output:** `models/lightgbm_model.pkl` | `reports/09_lightgbm_metrics.txt`

## Learning Objectives
- Understand how LightGBM differs from XGBoost (leaf-wise vs depth-wise growth)
- Apply is_unbalance=True for class imbalance handling
- Compare training speed and memory usage vs XGBoost
- Select champion model based on AUC-PR on validation set

## Business Context
LightGBM often trains 10-20x faster than XGBoost and uses less memory.
For the platform's eventual weekly model retraining on real data, speed matters.
But speed does not win -- AUC-PR on the validation set decides the champion.

## Concepts You Must Know Before Writing Code
1. What is leaf-wise growth? How does it differ from depth-wise (XGBoost)?
2. What does is_unbalance=True do differently than scale_pos_weight?
3. Why might LightGBM overfit more easily on small datasets?
4. How do you use callbacks in LightGBM for early stopping?
5. What criterion will you use to select XGBoost vs LightGBM as champion?

## Step 1 -- Answer the Five Questions
Write your answers in markdown before writing code.

## Step 2 -- Configure and Train
Train LGBMClassifier with: is_unbalance=True, max_depth=6, n_estimators=500, learning_rate=0.05.
Use early stopping on validation AUC-PR.

## Step 3 -- Evaluate
Same metrics as Notebooks 07 and 08. Add LightGBM column to the comparison table.

## Step 4 -- Champion Selection
Compare XGBoost vs LightGBM on:
- AUC-PR (validation)
- Recall at 5% FPR
- F1
- Training time

Select the champion. State your reasoning in a markdown cell.

## Definition of Done
- [ ] LightGBM trained with is_unbalance=True
- [ ] Comparison table updated: Rule Baseline vs LR vs XGBoost vs LightGBM
- [ ] Champion selected with written justification
- [ ] Model saved to models/lightgbm_model.pkl
- [ ] Metrics saved to reports/09_lightgbm_metrics.txt

In [1]:
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score, average_precision_score

X_train = pd.read_parquet(r"../data/processed\X_train.parquet")
X_test  = pd.read_parquet(r"../data/processed\X_test.parquet")
y_train = pd.read_parquet(r"../data/processed\y_train.parquet").squeeze()
y_test  = pd.read_parquet(r"../data/processed\y_test.parquet").squeeze()

model_lgbm = LGBMClassifier(
    max_depth=5,
    n_estimators=400,
    is_unbalance=True,
    learning_rate=0.1,
    random_state=42,
    verbosity=-1
)
model_lgbm.fit(X_train, y_train)

y_pred = model_lgbm.predict(X_test)
y_prob = model_lgbm.predict_proba(X_test)[:, 1]

f1 = f1_score(y_test, y_pred)
auc_pr = average_precision_score(y_test, y_prob)

print(f"F1:     {f1:.4f}")
print(f"AUC-PR: {auc_pr:.4f}")


F1:     0.0569
AUC-PR: 0.0241


In [2]:
model_lgbm = LGBMClassifier(
    max_depth=5,
    n_estimators=400,
    class_weight="balanced",
    learning_rate=0.1,
    random_state=42,
    verbosity=-1
)
model_lgbm.fit(X_train, y_train.astype(int), 
               eval_set=[(X_test, y_test.astype(int))])

y_pred = model_lgbm.predict(X_test)
y_prob = model_lgbm.predict_proba(X_test)[:, 1]

f1 = f1_score(y_test, y_pred)
auc_pr = average_precision_score(y_test, y_prob)

print(f"F1:     {f1:.4f}")
print(f"AUC-PR: {auc_pr:.4f}")


F1:     0.8791
AUC-PR: 0.8309
